## Your pipeline, part 2: model changes + final submission
HW2, notebook 8 (Kaggle GPU). Builds on 04 (best YOLO config) and 07 (best inference strategy: `A2+A3_wbf`).

**Two modes** (setting `ENSEMBLE` in the settings cell)
- `ENSEMBLE = False` (first run, ~4.5-5 h): trains, then writes `submission_08_A2_A3_wbf.csv` for the new model.
- `ENSEMBLE = True` (later run, ~35-45 min): **no training**. Needs the first run's output as an input; fuses the new model with
  `final_04.pt` by WBF and writes `submission_08_ensemble.csv`. Stops immediately if the first run's output is missing.

**Steps**
1. Model experiments on fold 0, same evaluator and fold as 04, so each step is one change:
   - `M1_e20_cos`: 04's best config, **8 -> 20 epochs with a cosine learning-rate schedule** (Bag of Freebies, arXiv:1902.04103).
   - `M2_1280_e20_cos`: M1 at **1280 px** (full resolution of the exported frames; small objects, arXiv:2207.14096).
2. Final model: the better of M1 / M2, retrained on all training frames -> `final_08.pt`.
3. Inference: single pass vs `A2+A3_wbf` (flips + 1.33x + tiles, Weighted Boxes Fusion; chosen in notebook 07) on the official val.
4. Test submission. All prediction passes are saved (`passes_*.pkl.gz`), so the ensemble run does not recompute them.

**Needs on Kaggle:** GPU (T4), **Internet ON**, inputs: the competition, the output of 01 (`vd_prep`), the output of the **finished 04**
(`final_04.pt`, `results_04_*.csv`). Optional: the output of 07 (only to print its numbers next to these). Ensemble run: also this notebook's first-run output.

Protections from 04: training and evaluation in separate processes, chunked prediction, a GPU memory check before training,
finished runs and passes are reused from attached outputs, results are saved after every step.

In [1]:
%pip install -q ultralytics

In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # less GPU memory fragmentation

import os, io, json, time, math, random, shutil, contextlib
from pathlib import Path
import numpy as np, pandas as pd, cv2, torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

SEED = 0
def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all()
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE -> turn on a GPU in the session options")

INPUT, WORK = Path("/kaggle/input"), Path("/kaggle/working")

def find_all(pattern, depth=5):
    # search attached inputs at increasing nesting depth (Kaggle nests inputs differently per source)
    for d in range(depth):
        hit = sorted(INPUT.glob("/".join(["*"] * d + [pattern])))
        if hit:
            return hit
    return []

def find(pattern):
    hit = find_all(pattern)
    if not hit:
        raise FileNotFoundError(f"{pattern} not found under {INPUT}: check the attached inputs")
    return hit[0]

PREP = find("vd_prep/config.json").parent            # output of notebook 01
CFG = json.load(open(PREP / "config.json"))
TEST_CSV = find("test_images.csv"); ROOT = TEST_CSV.parent
VAL_DIR = find("val/annotations_coco.json").parent
FOLD = CFG["folds"]                                   # train clip -> fold id (GroupKFold by scene, from notebook 01)
METRICS = ["mAP", "AP50", "AP_s", "AP_m", "AP_l", "ms_per_img"]

import gc, gzip, pickle
from multiprocessing import Pool
from torchvision.ops import batched_nms
from ultralytics import YOLO, settings
from ultralytics.utils import DEFAULT_CFG_DICT
settings.update({"wandb": False})
DEVICE = "cuda"

# ---- what to run
ENSEMBLE = False                    # False: train + new model's submission | True (later run): 2-model ensemble only, no training
SECOND_MODEL = "final_04.pt"        # second model of the ensemble

# ---- training (fold 0, same fold as 04's reference)
EPOCHS, BATCH = 20, 4
FOLDS_TO_RUN = [0]
VAL_STEP = 5
NB, TAG, MODEL_NAME, MODEL_CLS = "08", "yolo08", "YOLO11s pipeline", YOLO
COMMON = dict(epochs=EPOCHS, batch=BATCH, workers=4, device=0, seed=SEED, warmup_epochs=2, close_mosaic=5,
              plots=False, exist_ok=True)

# ---- inference (strategy chosen in notebook 07)
UP_SCALE, TILE_ZOOM, TILE_OVERLAP = 1.33, 1.5, 0.2
NMS_IOU, WBF_IOU, MAX_DET = 0.6, 0.55, 500
MERGE_WORKERS = 4                   # CPU processes for WBF
PREC = {"quantize": 16} if "quantize" in DEFAULT_CFG_DICT else {"half": True}   # fp16 (renamed in Ultralytics 8.4)

# ---- input checks: fail here, not after hours
have04 = bool(find_all("final_04.pt")) and bool(find_all("results_04_folds.csv"))
print("notebook 04 output:", "found" if have04 else "MISSING (needed for the reference row and the ensemble)")
if ENSEMBLE:
    assert find_all("final_08.pt") and find_all("results_08_folds.csv") and find_all("passes_08_test.pkl.gz"), \
        "ENSEMBLE = True needs the output of the first 08 run as an input (final_08.pt, results_08_folds.csv, passes_08_*)"
    assert find_all(SECOND_MODEL), f"ENSEMBLE = True needs {SECOND_MODEL} (attach the output of notebook 04)"
    print("ensemble mode: no training, only the ensemble")

GPU: Tesla T4
notebook 04 output: found


### Data

In [3]:
def load_gt(path):
    # COCO ground truth; ignored regions (iscrowd=1) are copied to all 8 classes so they ignore detections of any class
    d = json.load(open(path))
    crowd = [a for a in d["annotations"] if a.get("iscrowd")]
    d["annotations"] = [a for a in d["annotations"] if not a.get("iscrowd")]
    nid = max([a["id"] for a in d["annotations"]], default=0) + 1
    for a in crowd:
        for c in range(1, 9):
            d["annotations"].append({**a, "id": nid, "category_id": c, "iscrowd": 1, "area": a["bbox"][2] * a["bbox"][3]})
            nid += 1
    gt = COCO()
    gt.dataset = d
    with contextlib.redirect_stdout(io.StringIO()):
        gt.createIndex()
    return gt

def coco_eval(gt, dets, img_ids):
    # standard COCO bbox metrics, up to 500 detections per image (as in the leaderboard)
    if not dets:
        return dict(mAP=0.0, AP50=0.0, AP_s=0.0, AP_m=0.0, AP_l=0.0)
    with contextlib.redirect_stdout(io.StringIO()):
        E = COCOeval(gt, gt.loadRes(dets), "bbox")
        E.params.imgIds = list(img_ids)
        E.params.maxDets = [1, 10, 500]
        E.evaluate(); E.accumulate(); E.summarize()
    s = E.stats                                  # s[1], s[3:6] already use maxDets[-1] = 500
    prec = E.eval["precision"][:, :, :, 0, -1]   # [IoU, recall, class] for all sizes at 500 dets (pycocotools' s[0] hardcodes 100)
    prec = prec[prec > -1]
    return dict(mAP=float(prec.mean()) if prec.size else -1.0, AP50=s[1], AP_s=s[3], AP_m=s[4], AP_l=s[5])

def write_submission(dets, path):
    # one row per test image: "class score x y w h ..." (0-based class, pixels, top-left corner, max 500 boxes)
    test = pd.read_csv(TEST_CSV)
    by = {}
    for d in dets:
        by.setdefault(d["image_id"], []).append(d)
    rows = []
    for iid in test.image_id:
        top = sorted(by.get(iid, []), key=lambda d: -d["score"])[:500]
        rows.append(" ".join(f"{d['category_id'] - 1} {d['score']:.4f} " + " ".join(f"{v:.1f}" for v in d["bbox"]) for d in top))
    sub = pd.DataFrame({"image_id": test.image_id, "PredictionString": rows})
    sub.to_csv(path, index=False)
    sample = find_all("sample_submission.csv")      # not shipped with this competition's data: ids come from test_images.csv
    if sample:
        assert list(sub.image_id) == list(pd.read_csv(sample[0]).image_id), "image ids differ from sample_submission.csv"
    print(f"saved {path}: {len(sub)} rows, {sum(map(bool, rows))} with detections, {sum(len(r.split()) // 6 for r in rows)} boxes")

In [4]:
# training frames exported by notebook 01 (ignored regions already grayed out)
TJ = json.load(open(PREP / "train_coco.json"))
IMGS = {i["id"]: i for i in TJ["images"]}
GT_TRAIN = load_gt(PREP / "train_coco.json")

def fold_ids(f):
    # (train ids, held-out ids) for fold f; f=None -> all frames for training
    return ([i for i, m in IMGS.items() if FOLD[m["video"]] != f],
            [i for i, m in IMGS.items() if FOLD[m["video"]] == f])

def copy_parallel(src, dst, workers=16):
    # copy a folder file by file with many threads (on network storage the per-file latency dominates)
    from concurrent.futures import ThreadPoolExecutor
    files = [p for p in Path(src).rglob("*") if p.is_file()]
    def cp(p):
        t = Path(dst) / p.relative_to(src)
        t.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(p, t)
    with ThreadPoolExecutor(workers) as ex:
        list(ex.map(cp, files))

# /kaggle/input is network storage (~5 MB/s, ~50 ms per file): copy the training images once to local disk
IMG_ROOT = PREP / "images" / "train"
if str(PREP).startswith("/kaggle/input") and not ENSEMBLE:   # no training in the ensemble mode: no copy
    local = Path("/tmp/vd_prep/images/train")
    if not (local.parent / ".copied").exists():
        t0 = time.time()
        copy_parallel(IMG_ROOT, local)
        (local.parent / ".copied").touch()
        print(f"copied {sum(1 for _ in local.iterdir())} training images to local disk in {time.time() - t0:.0f} s")
    IMG_ROOT = local

def train_path(i):
    return IMG_ROOT / IMGS[i]["file_name"]

# official val: raw frames at original resolution, every VAL_STEP-th frame of each clip (for speed)
VAL_GT = load_gt(VAL_DIR / "annotations_coco.json")
_idx = {(p.parent.name, p.name): p for p in (VAL_DIR / "images").rglob("*.jpg")}
_v = pd.DataFrame(VAL_GT.dataset["images"])
_v = _v[(_v.groupby("video").frame_id.rank(method="first").astype(int) - 1) % VAL_STEP == 0]
VAL_ITEMS = [(int(r.id), _idx[(r.video, Path(r.file_name).name)]) for r in _v.itertuples()]
VAL_IDS = [i for i, _ in VAL_ITEMS]
TEST_ITEMS = [(r.image_id, ROOT / r.file_name) for r in pd.read_csv(TEST_CSV).itertuples()]
print(f"{len(IMGS)} train frames | {len(VAL_ITEMS)} val frames | {len(TEST_ITEMS)} test frames")

copied 4868 training images to local disk in 27 s
4868 train frames | 571 val frames | 3811 test frames


In [5]:
import sys, subprocess

DEVICE_ID = globals().get("DEVICE_ID", 0)
# Ultralytics finds labels by swapping images -> labels in the path, and writes a .cache next to them
IMG_DIR = IMG_ROOT                                          # local copy of the training images (data cell)
LBL_DIR = IMG_DIR.parent.parent / "labels" / "train"
if not LBL_DIR.exists():
    copy_parallel(PREP / "labels" / "train", LBL_DIR)
JOB_DIR = Path("/tmp/yolo_jobs") if str(WORK).startswith("/kaggle") else WORK / "yolo_jobs"   # image lists, yamls, job files
JOB_DIR.mkdir(parents=True, exist_ok=True)

def write_data(name, tr_ids, va_ids):
    # Ultralytics data yaml with explicit image lists, so folds are just different lists
    for part, ids in (("train", tr_ids), ("val", va_ids)):
        (JOB_DIR / f"{name}_{part}.txt").write_text("\n".join(str(IMG_DIR / IMGS[i]["file_name"]) for i in ids))
    q = lambda x: json.dumps(str(x))                        # quoted path, safe for Windows backslashes
    p = JOB_DIR / f"{name}.yaml"
    p.write_text(f"path: {q(JOB_DIR)}\ntrain: {q(JOB_DIR / (name + '_train.txt'))}\nval: {q(JOB_DIR / (name + '_val.txt'))}\nnames: {json.dumps(CFG['classes'])}\n")
    return str(p)

# Every training and every prediction runs in its own short-lived Python process. When it exits, all of its
# GPU memory is released, so nothing piles up between runs (inside one process it did, and the GPU ran out).
WORKER = JOB_DIR / "ultra_worker.py"
WORKER.write_text("""
import json, sys
job = json.load(open(sys.argv[1]))
import torch
from ultralytics import YOLO, RTDETR, settings
settings.update({"wandb": False})
Cls = {"YOLO": YOLO, "RTDETR": RTDETR}[job["cls"]]
out = {}
if job["kind"] == "train":
    model = Cls(job["build"])
    if job.get("load"):
        model = model.load(job["load"])                     # built from a yaml: transfer the pretrained weights
    model.train(**job["kwargs"])
else:
    from ultralytics.utils import DEFAULT_CFG_DICT
    kw = dict(job["kwargs"])
    if job.get("fp16"):                                     # newer versions renamed half=True to quantize=16
        kw.update({"quantize": 16} if "quantize" in DEFAULT_CFG_DICT else {"half": True})
    model = Cls(job["weights"])
    items = json.load(open(job["items"]))
    dets, ms, CHUNK = [], [], 16
    # small chunks: Ultralytics 8.4 loads a whole list of paths as ONE batch (ignores `batch`) -> out of memory
    for k in range(0, len(items), CHUNK):
        chunk = items[k:k + CHUNK]
        for (iid, _), r in zip(chunk, model.predict([p for _, p in chunk], stream=True, **kw)):
            b = r.boxes
            xywh = b.xyxy.cpu().numpy()
            xywh[:, 2:] -= xywh[:, :2]
            for box, s, c in zip(xywh.tolist(), b.conf.tolist(), b.cls.tolist()):
                dets.append([iid, int(c) + 1, *[round(v, 2) for v in box], round(s, 5)])
            ms.append(r.speed["inference"] + r.speed["postprocess"])
    out["dets"], out["ms"] = dets, sum(ms) / max(len(ms), 1)
if torch.cuda.is_available():
    out["peak_gb"] = torch.cuda.max_memory_reserved() / 1e9
json.dump(out, open(job["out"], "w"))
""")

def run_worker(job, tag):
    # runs one job in a fresh process, prints its log here, returns its result
    job = {**job, "out": str(JOB_DIR / f"{tag}.out.json")}
    jp = JOB_DIR / f"{tag}.job.json"
    jp.write_text(json.dumps(job))
    proc = subprocess.Popen([sys.executable, str(WORKER), str(jp)], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"{tag}: the worker process failed, see its log above")
    return json.loads(Path(job["out"]).read_text())

def fit(spec, name, tr_ids, va_ids, **hp):
    kw = {"data": write_data(name, tr_ids, va_ids), "project": str(WORK / "runs"), "name": name, **{**COMMON, **hp}}
    res = run_worker({**spec, "kind": "train", "kwargs": kw}, name)
    (WORK / "curves").mkdir(exist_ok=True)
    shutil.copy(WORK / "runs" / name / "results.csv", WORK / "curves" / f"{TAG}_{name}.csv")     # per-epoch val curve
    # last epoch, not "best": best is picked on the held-out fold
    return WORK / "runs" / name / "weights" / "last.pt", res.get("peak_gb", float("nan"))

def detect(weights, items, imgsz):
    # returns COCO-style detections in original pixel coordinates and model ms per image (inference + postprocess)
    tag = f"predict_{Path(weights).parent.parent.name}_{len(items)}"
    ip = JOB_DIR / f"{tag}.items.json"
    ip.write_text(json.dumps([[i, str(p)] for i, p in items]))
    kw = dict(imgsz=imgsz, conf=0.001, iou=0.6, max_det=500, batch=4, verbose=False, device=DEVICE_ID)
    res = run_worker({"cls": MODEL_CLS.__name__, "kind": "predict", "weights": str(weights), "items": str(ip),
                      "kwargs": kw, "fp16": True}, tag)
    dets = [{"image_id": d[0], "category_id": d[1], "bbox": d[2:6], "score": d[6]} for d in res["dets"]]
    return dets, res["ms"]

def eval_fold(weights, ids, imgsz):
    dets, ms = detect(weights, [(i, IMG_DIR / IMGS[i]["file_name"]) for i in ids], imgsz)
    return {**coco_eval(GT_TRAIN, dets, ids), "ms_per_img": ms}

### Experiments
Both start from 04's best config (read from its results) and change one thing at a time.

In [6]:
CONFIGS_04 = {"C0_imgsz640": dict(imgsz=640), "C1_imgsz960": dict(imgsz=960),
              "C2_imgsz960_lossgains": dict(imgsz=960, box=12.0, dfl=2.0)}           # the configs of notebook 04
s4 = find_all("results_04_summary.csv")
best04 = "C1_imgsz960"
if s4:
    fin = pd.read_csv(s4[0]).query("split == 'val'").config.iloc[0]
    best04 = fin.replace(" (final)", "")
base = CONFIGS_04[best04]
CONFIGS = {
    "M1_e20_cos":      {**base, "epochs": EPOCHS, "cos_lr": True},                  # longer training + cosine schedule
    "M2_1280_e20_cos": {**base, "epochs": EPOCHS, "cos_lr": True, "imgsz": 1280},   # + 1280 px
}
make_model = lambda cfg: {"cls": "YOLO", "build": "yolo11s.pt"}     # built inside the worker process
hp_of = lambda cfg: dict(cfg)
print("04's best config:", best04, base)
for k, v in CONFIGS.items():
    print(f"  {k}: {v}")

04's best config: C2_imgsz960_lossgains {'imgsz': 960, 'box': 12.0, 'dfl': 2.0}
  M1_e20_cos: {'imgsz': 960, 'box': 12.0, 'dfl': 2.0, 'epochs': 20, 'cos_lr': True}
  M2_1280_e20_cos: {'imgsz': 1280, 'box': 12.0, 'dfl': 2.0, 'epochs': 20, 'cos_lr': True}


### GPU memory check (~5 min)
One epoch on the 64 most crowded frames with the heaviest config (1280 px), then a prediction on them at the same size.
Skipped in the ensemble mode (no training).

In [7]:
if not ENSEMBLE:
    n_obj = {}
    for a in TJ["annotations"]:
        if not a.get("iscrowd"):
            n_obj[a["image_id"]] = n_obj.get(a["image_id"], 0) + 1
    dense = sorted(IMGS, key=lambda i: -n_obj.get(i, 0))[:64]
    heavy = max(CONFIGS.values(), key=lambda c: c["imgsz"])
    kw = {"data": write_data("memcheck", dense, dense[:16]), "project": str(WORK / "runs"), "name": "memcheck",
          **{**COMMON, **hp_of(heavy), "epochs": 1, "warmup_epochs": 0, "close_mosaic": 0}}
    res = run_worker({**make_model(heavy), "kind": "train", "kwargs": kw}, "memcheck")
    mc_dets, _ = detect(WORK / "runs" / "memcheck" / "weights" / "last.pt", [(i, IMG_DIR / IMGS[i]["file_name"]) for i in dense], heavy["imgsz"])
    print(f"prediction check: {len(dense)} frames -> {len(mc_dets)} boxes")
    shutil.rmtree(WORK / "runs" / "memcheck", ignore_errors=True)
    peak, total = res.get("peak_gb", 0.0), torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"peak GPU memory {peak:.1f} of {total:.1f} GB")
    assert peak < 0.85 * total, "too close to the GPU memory limit: lower BATCH in the settings cell and run again"
else:
    print("ensemble mode: memory check skipped")

1 epochs completed in 0.002 hours.
prediction check: 64 frames -> 32000 boxes
peak GPU memory 5.7 of 15.6 GB


### Cross-validation runs (fold 0)

In [8]:
# every config is trained on the same folds (cross-validation over clips) and scored on the held-out clips.
# Results are saved after every run; runs already in results_{NB}_folds.csv (here, or in an attached
# earlier version's output) are skipped, so a crash does not cost the finished runs again.
prev = [p for p in [WORK / f"results_{NB}_folds.csv"] if p.exists()] + find_all(f"results_{NB}_folds.csv")
rows = pd.read_csv(prev[0]).to_dict("records") if prev else []
done = {(r["config"], int(r["fold"])) for r in rows}
if done:
    print("reusing finished runs from", prev[0], ":", sorted(done))
    pd.DataFrame(rows).to_csv(WORK / f"results_{NB}_folds.csv", index=False)     # keep them in this version's output too
(WORK / "curves").mkdir(exist_ok=True)
for name, cfg in CONFIGS.items():
    for f in FOLDS_TO_RUN:
        if (name, f) in done:
            for p in find_all(f"curves/{TAG}_{name}_f{f}.csv"):        # keep its training curve for notebook 06
                if not (WORK / "curves" / p.name).exists():
                    shutil.copy(p, WORK / "curves" / p.name)
            continue
        tr_ids, va_ids = fold_ids(f)
        t0 = time.time()
        run = f"{name}_f{f}"
        trained = [p for p in [WORK / "runs" / run / "weights" / "last.pt"] if p.exists()] + find_all(f"runs/{run}/weights/last.pt")
        curve = [p for p in [WORK / "runs" / run / "results.csv"] if p.exists()] + find_all(f"runs/{run}/results.csv")
        if trained and curve:                                   # trained in an earlier, interrupted version: only evaluate
            print("reusing trained weights", trained[0])
            w, peak = trained[0], float("nan")
            if curve[0] != WORK / "curves" / f"{TAG}_{run}.csv":
                shutil.copy(curve[0], WORK / "curves" / f"{TAG}_{run}.csv")
        else:
            w, peak = fit(make_model(cfg), run, tr_ids, va_ids, **hp_of(cfg))
        m = eval_fold(w, va_ids, cfg["imgsz"])
        rows.append({"config": name, "fold": f, **m, "train_min": (time.time() - t0) / 60, "peak_gb": peak})
        pd.DataFrame(rows).to_csv(WORK / f"results_{NB}_folds.csv", index=False)
        print(name, "fold", f, {k: round(v, 4) for k, v in m.items()}, f"| peak GPU memory of the training: {peak:.1f} GB")
folds_df = pd.DataFrame(rows)
folds_df = folds_df[folds_df.config.isin(CONFIGS) & folds_df.fold.isin(FOLDS_TO_RUN)]
display(folds_df.groupby("config")[METRICS].agg(["mean", "std"]).round(4))

20 epochs completed in 0.709 hours.
M1_e20_cos fold 0 {'mAP': 0.1881, 'AP50': np.float64(0.3623), 'AP_s': np.float64(0.0742), 'AP_m': np.float64(0.2396), 'AP_l': np.float64(0.5071), 'ms_per_img': 6.4516} | peak GPU memory of the training: 3.1 GB
20 epochs completed in 1.209 hours.
M2_1280_e20_cos fold 0 {'mAP': 0.1942, 'AP50': np.float64(0.3826), 'AP_s': np.float64(0.0868), 'AP_m': np.float64(0.2596), 'AP_l': np.float64(0.4481), 'ms_per_img': 10.7229} | peak GPU memory of the training: 5.5 GB


mAP        AP50        AP_s        AP_m        AP_l      \
                   mean std    mean std    mean std    mean std    mean std   
config                                                                        
M1_e20_cos       0.1881 NaN  0.3623 NaN  0.0742 NaN  0.2396 NaN  0.5071 NaN   
M2_1280_e20_cos  0.1942 NaN  0.3826 NaN  0.0868 NaN  0.2596 NaN  0.4481 NaN   

                ms_per_img      
                      mean std  
config                          
M1_e20_cos          6.4516 NaN  
M2_1280_e20_cos    10.7229 NaN

In [9]:
# reference: 04's best config on the same fold (8 epochs), then the two changes
cmp = []
r4 = find_all("results_04_folds.csv")
if r4:
    d4 = pd.read_csv(r4[0])
    d4 = d4[(d4.config == best04) & d4.fold.isin(FOLDS_TO_RUN)]
    if len(d4):
        cmp.append({"config": f"04 {best04} (8 epochs)", **d4[METRICS].mean().to_dict()})
cmp += [{"config": c, **folds_df[folds_df.config == c][METRICS].mean().to_dict()} for c in CONFIGS if (folds_df.config == c).any()]
display(pd.DataFrame(cmp).set_index("config").round(4))

,mAP,AP50,AP_s,AP_m,AP_l,ms_per_img
config,,,,,,
04 C2_imgsz960_lossgains (8 epochs),0.2018,0.3883,0.0835,0.2678,0.4662,6.8313
M1_e20_cos,0.1881,0.3623,0.0742,0.2396,0.5071,6.4516
M2_1280_e20_cos,0.1942,0.3826,0.0868,0.2596,0.4481,10.7229


**Observations:**
- Longer training does not help on the held-out fold: 20 epochs with a cosine schedule gives 0.188 vs 0.202 for 04's 8-epoch model. AP_large improves (0.466 -> 0.507) but AP_small drops (0.084 -> 0.074), which points to overfitting on the training clips.
- 1280 px (20 epochs) recovers part of it: 0.194 with the best AP_small of the three (0.087), but still below 04's 8-epoch model on this fold.
- Cost: 0.71 h (960 px) and 1.21 h (1280 px) per run, vs 0.31 h for 04, and 10.7 vs 6.5 ms per image at 1280 px.
- Only one fold, so these differences (~0.01) are about the size of the fold-to-fold noise seen in 04.

### Final model: best of M1 / M2 on all training frames

In [10]:
best = folds_df.groupby("config").mAP.mean().idxmax()
FINAL_CFG = CONFIGS[best]
print("best config on fold", FOLDS_TO_RUN, ":", best, FINAL_CFG)
done_final = [p for p in [WORK / "final_08.pt"] if p.exists()] + find_all("final_08.pt")
if done_final:                                              # trained in an earlier run of this notebook
    W_FINAL = done_final[0]
    print("reusing", W_FINAL)
else:
    run = "final_08"
    trained = [p for p in [WORK / "runs" / run / "weights" / "last.pt"] if p.exists()] + find_all(f"runs/{run}/weights/last.pt")
    if trained and ([p for p in [WORK / "runs" / run / "results.csv"] if p.exists()] + find_all(f"runs/{run}/results.csv")):
        w = trained[0]
        print("reusing trained weights", w)
    else:
        tr_all, _ = fold_ids(None)
        w, peak = fit(make_model(FINAL_CFG), run, tr_all, fold_ids(0)[1][:200], **hp_of(FINAL_CFG))  # 200 held-out frames only feed the log
        print(f"final model trained | peak GPU memory {peak:.1f} GB")
    W_FINAL = WORK / "final_08.pt"
    shutil.copy(w, W_FINAL)
if W_FINAL != WORK / "final_08.pt":
    shutil.copy(W_FINAL, WORK / "final_08.pt")              # keep it in this version's output too

best config on fold [0] : M2_1280_e20_cos {'imgsz': 1280, 'box': 12.0, 'dfl': 2.0, 'epochs': 20, 'cos_lr': True}
20 epochs completed in 1.442 hours.
final model trained | peak GPU memory 5.3 GB


### Inference: prediction passes and merging (from notebook 07)
Passes of a model on a set of frames are computed once and saved (`passes_<model>_<split>.pkl.gz`); an attached earlier output is reused.

In [11]:
def load_model(path):
    # model + the image size it was trained at
    ck = torch.load(path, map_location="cpu", weights_only=False)
    imgsz = int(np.max(ck.get("train_args", {}).get("imgsz", 960)))
    del ck
    return YOLO(str(path)), imgsz

def run(model, images, imgsz):
    # one model call on a few images (Ultralytics 8.4 puts a whole list into one batch); one (N, 6) array per image
    res = model.predict(images, imgsz=imgsz, conf=0.001, iou=0.6, max_det=MAX_DET, verbose=False, device=DEVICE_ID, **PREC)
    return [np.c_[r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.boxes.cls.cpu().numpy()].astype(np.float32) for r in res]

def unflip(b, width):
    # boxes predicted on a horizontally flipped image -> original orientation
    b = b.copy(); b[:, [0, 2]] = width - b[:, [2, 0]]
    return b

def nms(b, iou):
    # class-wise NMS, best first, at most MAX_DET boxes (main process only)
    if not len(b):
        return b
    t = torch.from_numpy(b)
    keep = batched_nms(t[:, :4], t[:, 4], t[:, 5].long(), iou).numpy()
    return b[keep[:MAX_DET]]

def tile_grid(W, H):
    # overlapping crops of side max(W, H) / TILE_ZOOM (clipped to the frame), at least TILE_OVERLAP overlap
    T = int(max(W, H) / TILE_ZOOM)
    tw, th = min(T, W), min(T, H)
    def starts(size, t):
        n = max(1, math.ceil((size - t) / (t * (1 - TILE_OVERLAP))) + 1) if size > t else 1
        return np.linspace(0, size - t, n).round().astype(int)
    return [(x, y, x + tw, y + th) for y in starts(H, th) for x in starts(W, tw)]

ALL_VIEWS = ["full", "full_flip", "up", "up_flip", "tiles", "tiles_flip"]

def predict_views(model, imgsz, img):
    # all six passes of one frame, boxes in original pixel coordinates
    H, W = img.shape[:2]
    up = int(round(imgsz * UP_SCALE / 32) * 32)
    flip = np.ascontiguousarray(img[:, ::-1])
    out = {"full": run(model, [img], imgsz)[0], "full_flip": unflip(run(model, [flip], imgsz)[0], W),
           "up": run(model, [img], up)[0], "up_flip": unflip(run(model, [flip], up)[0], W)}
    grid = tile_grid(W, H)
    for name in ("tiles", "tiles_flip"):
        crops = [img[y0:y1, x0:x1] for x0, y0, x1, y1 in grid]
        if name == "tiles_flip":
            crops = [np.ascontiguousarray(c[:, ::-1]) for c in crops]
        parts = []
        for (x0, y0, x1, y1), b in zip(grid, run(model, crops, imgsz)):
            if name == "tiles_flip":
                b = unflip(b, x1 - x0)
            # drop boxes cut by an inner crop edge: the neighbouring crop (>= 20 % overlap) or the full pass has the whole object
            e = 2
            cut = (((b[:, 0] <= e) & (x0 > 0)) | ((b[:, 2] >= x1 - x0 - e) & (x1 < W)) |
                   ((b[:, 1] <= e) & (y0 > 0)) | ((b[:, 3] >= y1 - y0 - e) & (y1 < H)))
            b = b[~cut].copy(); b[:, [0, 2]] += x0; b[:, [1, 3]] += y0
            parts.append(b)
        out[name] = nms(np.concatenate(parts), NMS_IOU)       # remove duplicates from overlapping crops
    return out

def passes(model_path, tag, items):
    # prediction passes of one model on a set of frames; reused from this or an attached output, else computed and saved
    fname = f"passes_{tag}.pkl.gz"
    hit = [p for p in [WORK / fname] if p.exists()] + find_all(fname)
    if hit:
        with gzip.open(hit[0], "rb") as f:
            cache = pickle.load(f)
        if set(cache) == {i for i, _ in items}:
            print(f"reusing {hit[0]}")
            return cache
        print(f"{hit[0]} covers other frames, recomputing")
    model, imgsz = load_model(model_path)
    print(f"{tag}: {Path(model_path).name} trained at {imgsz} px, {len(items)} frames")
    cache = {iid: predict_views(model, imgsz, cv2.imread(str(p))) for iid, p in tqdm(items, desc=tag)}
    del model; gc.collect(); torch.cuda.empty_cache()
    with gzip.open(WORK / fname, "wb") as f:
        pickle.dump(cache, f, protocol=4)
    return cache

In [12]:
def wbf(views, iou_thr=WBF_IOU):
    # views: list of (N, 6) arrays, one per pass; returns fused (M, 6), best first, at most MAX_DET
    n_views = len(views)
    allb = np.concatenate([v for v in views if len(v)] or [np.zeros((0, 6), np.float32)])
    out = []
    for c in np.unique(allb[:, 5]):
        b = allb[allb[:, 5] == c]
        b = b[np.argsort(-b[:, 4])]
        acc = np.zeros((len(b), 4)); fb = np.zeros((len(b), 4)); w = np.zeros(len(b)); cnt = np.zeros(len(b)); m = 0
        for x1, y1, x2, y2, s, _ in b:
            if m:
                f = fb[:m]
                ix = np.clip(np.minimum(f[:, 2], x2) - np.maximum(f[:, 0], x1), 0, None)
                iy = np.clip(np.minimum(f[:, 3], y2) - np.maximum(f[:, 1], y1), 0, None)
                inter = ix * iy
                iou = inter / ((f[:, 2] - f[:, 0]) * (f[:, 3] - f[:, 1]) + (x2 - x1) * (y2 - y1) - inter + 1e-9)
                j = int(np.argmax(iou))
                if iou[j] > iou_thr:
                    acc[j] += s * np.array([x1, y1, x2, y2]); w[j] += s; cnt[j] += 1
                    fb[j] = acc[j] / w[j]
                    continue
            acc[m] = s * np.array([x1, y1, x2, y2]); w[m] = s; cnt[m] = 1; fb[m] = (x1, y1, x2, y2); m += 1
        score = w[:m] / cnt[:m] * np.minimum(cnt[:m], n_views) / n_views
        out.append(np.c_[fb[:m], score, np.full(m, c)])
    if not out:
        return np.zeros((0, 6), np.float32)
    res = np.concatenate(out).astype(np.float32)
    return res[np.argsort(-res[:, 4])][:MAX_DET]

def merge(views, how):
    if how is None:                                         # single pass: as predicted
        return views[0][:MAX_DET]
    if how == "nms":
        return nms(np.concatenate(views), NMS_IOU)
    return wbf(views)

STRATEGIES = {                                              # name: (views, merge)
    "S0_single_pass":   (["full"], None),
    "A1_upscaled_pass": (["up"], None),
    "A2_tta_nms":       (["full", "full_flip", "up", "up_flip"], "nms"),
    "A2_tta_wbf":       (["full", "full_flip", "up", "up_flip"], "wbf"),
    "A3_tiles_nms":     (["full", "tiles"], "nms"),
    "A3_tiles_wbf":     (["full", "tiles"], "wbf"),
    "A2+A3_wbf":        (["full", "full_flip", "up", "up_flip", "tiles", "tiles_flip"], "wbf"),
}
ALL_VIEWS = ["full", "full_flip", "up", "up_flip", "tiles", "tiles_flip"]

def merge_task(args):
    # one image, one strategy (runs in a worker process)
    iid, views, how = args
    t0 = time.perf_counter()
    b = merge(views, how)
    return iid, b, 1000 * (time.perf_counter() - t0)

def merge_all(cache, views, how):
    # merge the cached passes of every frame; returns detections (COCO style) and merge ms per frame
    tasks = [(iid, [cache[iid][v] for v in views], how) for iid in cache]
    # worker processes only for WBF (pure NumPy). NMS uses PyTorch, which can hang inside forked processes
    # once its thread pool was used in the main process; NMS takes only seconds here anyway.
    if MERGE_WORKERS and how == "wbf":
        with Pool(MERGE_WORKERS) as p:
            res = p.map(merge_task, tasks, chunksize=8)
    else:
        res = [merge_task(t) for t in tasks]
    dets = [{"image_id": iid, "category_id": int(r[5]) + 1, "bbox": [float(r[0]), float(r[1]), float(r[2] - r[0]), float(r[3] - r[1])],
             "score": float(r[4])} for iid, b, _ in res for r in b]
    return dets, float(np.mean([t for _, _, t in res]))

### Val: new model, single pass vs A2+A3 (WBF)

In [13]:
val_new = passes(W_FINAL, "08_val", VAL_ITEMS)
rows_val = []
for name, (views, how) in {"single_pass": (["full"], None), "A2+A3_wbf": (ALL_VIEWS, "wbf")}.items():
    dets, _ = merge_all(val_new, views, how)
    rows_val.append({"config": f"final_08 {name}", **coco_eval(VAL_GT, dets, VAL_IDS)})
    print(rows_val[-1])
r7 = find_all("results_07_inference.csv")
if r7:                                                      # the same two strategies with 04's model (notebook 07)
    d7 = pd.read_csv(r7[0]).set_index("strategy")
    for s, label in (("S0_single_pass", "single_pass"), ("A2+A3_wbf", "A2+A3_wbf")):
        if s in d7.index:
            rows_val.append({"config": f"final_04 {label} (notebook 07)", **d7.loc[s, ["mAP", "AP50", "AP_s", "AP_m", "AP_l"]].to_dict()})
display(pd.DataFrame(rows_val).set_index("config").round(4))

08_val: final_08.pt trained at 1280 px, 571 frames
{'config': 'final_08 single_pass', 'mAP': 0.1821556214359974, 'AP50': np.float64(0.36977009554811063), 'AP_s': np.float64(0.10685782030643098), 'AP_m': np.float64(0.18734663034160448), 'AP_l': np.float64(0.28235542688505333)}
{'config': 'final_08 A2+A3_wbf', 'mAP': 0.20672507032681892, 'AP50': np.float64(0.4161976629141665), 'AP_s': np.float64(0.129048155460224), 'AP_m': np.float64(0.21288119544202805), 'AP_l': np.float64(0.3078354763611108)}


,mAP,AP50,AP_s,AP_m,AP_l
config,,,,,
final_08 single_pass,0.1822,0.3698,0.1069,0.1873,0.2824
final_08 A2+A3_wbf,0.2067,0.4162,0.1290,0.2129,0.3078
final_04 single_pass (notebook 07),0.1699,0.3410,0.0981,0.1661,0.3023
final_04 A2+A3_wbf (notebook 07),0.2045,0.4058,0.1644,0.2058,0.3279


### Test submission: new model with A2+A3 (WBF)

In [14]:
test_new = passes(W_FINAL, "08_test", TEST_ITEMS)           # computed in the first run, loaded in the ensemble run
if not ENSEMBLE:
    dets, _ = merge_all(test_new, ALL_VIEWS, "wbf")
    write_submission(dets, WORK / "submission_08_A2_A3_wbf.csv")

08_test: final_08.pt trained at 1280 px, 3811 frames
saved /kaggle/working/submission_08_A2_A3_wbf.csv: 3811 rows, 3811 with detections, 1598740 boxes


### Ensemble (only with `ENSEMBLE = True`)
The new model's six passes and the second model's six passes are fused in one WBF step (12 passes).

In [15]:
if ENSEMBLE:
    sec = find(SECOND_MODEL)
    tag2 = Path(SECOND_MODEL).stem
    both = lambda a, b: {i: {**{"a_" + v: a[i][v] for v in ALL_VIEWS}, **{"b_" + v: b[i][v] for v in ALL_VIEWS}} for i in a}
    VIEWS2 = ["a_" + v for v in ALL_VIEWS] + ["b_" + v for v in ALL_VIEWS]

    val_ens = both(val_new, passes(sec, f"{tag2}_val", VAL_ITEMS))
    dets, _ = merge_all(val_ens, VIEWS2, "wbf")
    rows_val.append({"config": f"ensemble final_08 + {tag2} A2+A3_wbf", **coco_eval(VAL_GT, dets, VAL_IDS)})
    display(pd.DataFrame(rows_val).set_index("config").round(4))
    single = next(r["mAP"] for r in rows_val if r["config"] == "final_08 A2+A3_wbf")
    print("ensemble vs new model alone on val:", round(rows_val[-1]["mAP"] - single, 4),
          "-> submit the ensemble" if rows_val[-1]["mAP"] > single else "-> the new model alone is better, keep submission_08_A2_A3_wbf.csv")

    test_ens = both(test_new, passes(sec, f"{tag2}_test", TEST_ITEMS))
    dets, _ = merge_all(test_ens, VIEWS2, "wbf")
    write_submission(dets, WORK / "submission_08_ensemble.csv")

### Summary

In [16]:
out = folds_df.groupby("config")[METRICS].mean().reset_index().assign(split="cv")
out = pd.concat([out, pd.DataFrame(rows_val).assign(split="val")], ignore_index=True)
out.insert(0, "model", MODEL_NAME)
out.to_csv(WORK / f"results_{NB}_summary.csv", index=False)
display(out.round(4))

,model,config,mAP,AP50,AP_s,AP_m,AP_l,ms_per_img,split
0,YOLO11s pipeline,M1_e20_cos,0.1881,0.3623,0.0742,0.2396,0.5071,6.4516,cv
1,YOLO11s pipeline,M2_1280_e20_cos,0.1942,0.3826,0.0868,0.2596,0.4481,10.7229,cv
2,YOLO11s pipeline,final_08 single_pass,0.1822,0.3698,0.1069,0.1873,0.2824,NaN,val
3,YOLO11s pipeline,final_08 A2+A3_wbf,0.2067,0.4162,0.1290,0.2129,0.3078,NaN,val
4,YOLO11s pipeline,final_04 single_pass (notebook 07),0.1699,0.3410,0.0981,0.1661,0.3023,NaN,val
5,YOLO11s pipeline,final_04 A2+A3_wbf (notebook 07),0.2045,0.4058,0.1644,0.2058,0.3279,NaN,val


**Observations:**
- The model change helps a single pass on val (0.182 vs 0.170 for final_04), but the six-pass inference adds less on top of it (+0.025 vs +0.035). At 1280 px the model already sees small objects larger, so upscaling and tiles add less: AP_small with all passes is 0.129 vs 0.164 for final_04.
- Combined, final_08 + A2+A3 is only +0.002 better on val (0.207 vs 0.205) and **worse on the leaderboard (0.239 vs 0.269)**. The val gain did not transfer: the longer-trained model generalises worse to the test clips.
- The ensemble mode was not run here; the two models were fused later in notebook 09.